In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [22]:
path = r"C:\Users\chris\OneDrive\Desktop\Programming\Trading\prediction markets\crypto\brti_data_collection\data\brti_price_data.parquet"

df = pd.read_parquet(path)
df.info()
df.set_index('timestamp_ms')
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90291 entries, 0 to 90290
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   timestamp_est   90291 non-null  datetime64[ns]
 1   timestamp_ms    90291 non-null  int64         
 2   brti_price      90291 non-null  float64       
 3   simple_average  90291 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1)
memory usage: 2.8 MB


,timestamp_est,timestamp_ms,brti_price,simple_average
0,2025-06-17 12:21:33.719,1750177293719,104276.83,104276.830000
1,2025-06-17 12:21:34.030,1750177294030,104292.41,104284.620000
2,2025-06-17 12:21:34.988,1750177294988,104294.71,104287.983333
3,2025-06-17 12:21:35.956,1750177295956,104283.55,104286.875000
4,2025-06-17 12:21:36.978,1750177296978,104285.49,104286.598000


In [159]:
# time to 2pm in seconds
from datetime import datetime, timedelta
import pytz

def time_until_2pm():
    # Get current time in EST (since your data uses EST)
    est = pytz.timezone('US/Eastern')
    now = datetime.now(est)
    
    # Create today's 2 PM timestamp
    today_2pm = now.replace(hour=14, minute=0, second=0, microsecond=0)
    
    # If it's already past 2 PM, get tomorrow's 2 PM
    if now > today_2pm:
        today_2pm = today_2pm + timedelta(days=1)
    
    # Calculate time difference in seconds
    seconds_until_2pm = (today_2pm - now).total_seconds()
    
    return int(seconds_until_2pm)

# Use it
seconds = int(time_until_2pm())
print(f"Seconds until 2 PM EST: {seconds}")


Seconds until 2 PM EST: 2464


In [149]:
def bounded_binary_pricing(df, current_price, strike, market_mid=None):
    """
    Simple model that stays within 5c of reality
    """
    
    df = pd.read_parquet(r"C:\Users\chris\OneDrive\Desktop\Programming\Trading\prediction markets\crypto\brti_data_collection\data\brti_price_data.parquet")
    df['log_return'] = np.log(df['brti_price'] / df['brti_price'].shift(1))

    # 1. Base probability from moneyness
    moneyness = (strike - current_price) / current_price
    
    expiry_minutes = time_until_2pm()
    # Simple sigmoid function - works surprisingly well
    base_prob = 1 / (1 + np.exp(moneyness * 50))  # 50 is sensitivity parameter
    
    # 2. Time decay (options worth less as time runs out)
    time_factor = np.sqrt(expiry_minutes / 60)  # Square root time scaling
    
    # 3. Volatility adjustment (simple but effective)
    recent_vol = df['log_return'].tail(300).std()  # Last 5 minutes
    vol_adjustment = min(recent_vol * 10, 0.15)  # Cap at 15%
    
    # 4. Momentum component
    momentum = df['log_return'].tail(60).mean()  # Last minute
    momentum_adj = np.tanh(momentum * 1000) * 0.1  # Small adjustment
    
    # Combine everything
    fair_value = (base_prob * time_factor + 
                 vol_adjustment + 
                 momentum_adj)
    
    # 5. Bounds checking - this is the key part
    fair_value = np.clip(fair_value, 0.05, 0.95)
    
    # 6. Market calibration (if you have market data)
    if market_mid is not None:
        # Don't drift too far from market
        max_deviation = 0.05  # 5 cents
        
        if abs(fair_value - market_mid) > max_deviation:
            # Pull back toward market
            if fair_value > market_mid:
                fair_value = market_mid + max_deviation
            else:
                fair_value = market_mid - max_deviation
    
    return fair_value

In [158]:
bounded_binary_pricing(brti_data, 104086, 104000)

0.42696528157109676

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

def black_scholes_binary(df, S, strike, expiry_seconds, option_type='call', risk_free_rate=0.00):
    """
    Price binary options using Black-Scholes with 5-minute rolling volatility
    
    Parameters:
    -----------
    df : DataFrame
        Must contain 'brti_price' and 'log_return' columns
    strike : float
        Strike price
    expiry_seconds : int
        Time to expiry in seconds
    option_type : str
        'call' or 'put'
    risk_free_rate : float
        Annual risk-free rate
        
    Returns:
    --------
    float : Binary option price
    """
    # Current price
    K = strike
    
    # Time to expiry in years
    T = expiry_seconds / (365.25 * 24 * 3600)
    
    # Risk-free rate
    r = risk_free_rate
    
    # Calculate volatility from last 5 minutes (300 seconds) of data
    vol_window = min(300, len(df))  # 5 minutes or all available data
    returns1 = df['log_return'].tail(60).dropna()
    returns2 = df['log_return'].tail(300).dropna()
    returns3 = df['log_return'].tail(600).dropna()
    returns4 = df['log_return'].tail(1200).dropna()


    df['rbv_sum1'] = df['abs_return_product'].rolling(window=600, min_periods=600//2).sum()
    df['rbv_sum2'] = df['abs_return_product'].rolling(window=1200, min_periods=1200//2).sum()
    df['rbv_sum3'] = df['abs_return_product'].rolling(window=2400, min_periods=2400//2).sum()
    df['rbv_sum4'] = df['abs_return_product'].rolling(window=4800, min_periods=4800//2).sum()

    # Apply the (π/2) multiplier and annualize
    rbv1 = np.sqrt((np.pi / 2) * df['rbv_sum1'] * (365.25 * 24 * 60 * 60))  # Annualized
    rbv2 = np.sqrt((np.pi / 2) * df['rbv_sum2'] * (365.25 * 24 * 60 * 60))  # Annualized
    rbv3 = np.sqrt((np.pi / 2) * df['rbv_sum3'] * (365.25 * 24 * 60 * 60))  # Annualized
    rbv4 = np.sqrt((np.pi / 2) * df['rbv_sum4'] * (365.25 * 24 * 60 * 60))  # Annualized


    # Annualize the volatility (seconds to years)
    vol1 = returns1.std() * np.sqrt(365.25 * 24 * 3600)
    vol2 = returns2.std() * np.sqrt(365.25 * 24 * 3600)
    vol3 = returns3.std() * np.sqrt(365.25 * 24 * 3600)
    vol4 = returns4.std() * np.sqrt(365.25 * 24 * 3600)

    # take the average of the 4 volatilities
    vol = (vol1 + vol2 + vol3 + vol4) / 4

    rbv = (rbv1 + rbv2 + rbv3 + rbv4) / 4
    
    # Handle edge cases
    if T <= 0:
        # At expiry
        if option_type.lower() == 'call':
            return 1.0 if S > K else 0.0
        else:
            return 1.0 if S < K else 0.0
    
    if vol <= 0:
        vol = 0.01  # Minimum volatility
    
    # Black-Scholes d2 (the key parameter for binary options)
    d2 = (np.log(S / K) + (r - 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    
    # Binary option price
    if option_type.lower() == 'call':
        # Probability of finishing above strike
        prob = norm.cdf(d2)
    elif option_type.lower() == 'put':
        # Probability of finishing below strike
        prob = norm.cdf(-d2)
    else:
        raise ValueError("option_type must be 'call' or 'put'")
    
    # Discount to present value
    discount_factor = np.exp(-r * T)
    price = prob * discount_factor
    
    return price

def black_scholes_binary_with_greeks(df, strike, expiry_seconds, option_type='call', risk_free_rate=0.05):
    """
    Price binary option and calculate Greeks
    """
    S = df['brti_price'].iloc[-1]
    K = strike
    T = expiry_seconds / (365.25 * 24 * 3600)
    r = risk_free_rate
    
    # Volatility calculation (same as above)
    vol_window = min(300, len(df))
    returns = df['log_return'].tail(vol_window).dropna()
    
    if len(returns) < 10:
        if 'realized_var' in df.columns:
            vol = np.sqrt(df['realized_var'].tail(60).mean() * 365.25 * 24)
        else:
            vol = 0.5
    else:
        vol = returns.std() * np.sqrt(365.25 * 24 * 3600)
    
    # Handle edge cases
    if T <= 0:
        if option_type.lower() == 'call':
            return {'price': 1.0 if S > K else 0.0, 'delta': 0.0, 'gamma': 0.0, 'theta': 0.0, 'vega': 0.0}
        else:
            return {'price': 1.0 if S < K else 0.0, 'delta': 0.0, 'gamma': 0.0, 'theta': 0.0, 'vega': 0.0}
    
    if vol <= 0:
        vol = 0.01
    
    # Black-Scholes parameters
    d1 = (np.log(S / K) + (r + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    
    # Price
    discount_factor = np.exp(-r * T)
    
    if option_type.lower() == 'call':
        price = norm.cdf(d2) * discount_factor
        delta_sign = 1
    else:
        price = norm.cdf(-d2) * discount_factor
        delta_sign = -1
    
    # Greeks
    try:
        # Delta (sensitivity to underlying price)
        delta = delta_sign * norm.pdf(d2) * discount_factor / (S * vol * np.sqrt(T))
        
        # Gamma (second derivative wrt underlying)
        gamma = -delta_sign * norm.pdf(d2) * discount_factor * d1 / (S**2 * vol**2 * T)
        
        # Theta (time decay) - convert to per day
        theta = (delta_sign * norm.pdf(d2) * discount_factor * 
                (d1 / (2 * T) + r) / (365.25 * 24 * 3600))  # Per second
        
        # Vega (sensitivity to volatility) 
        vega = -delta_sign * norm.pdf(d2) * discount_factor * d1 / vol
        
    except (ZeroDivisionError, OverflowError):
        delta = gamma = theta = vega = 0.0
    
    return {
        'price': price,
        'delta': delta,
        'gamma': gamma, 
        'theta': theta,
        'vega': vega,
        'implied_vol': vol,
        'time_to_expiry': T,
        'moneyness': S / K
    }

def quick_binary_price(df, strike, expiry_seconds, option_type='call'):
    """
    Ultra-fast binary pricing for high-frequency use
    """
    S = df['brti_price'].iloc[-1]
    T = expiry_seconds / (365.25 * 24 * 3600)
    
    # Quick vol estimate (last 60 data points)
    returns = df['log_return'].tail(60).dropna()
    if len(returns) > 5:
        vol = returns.std() * np.sqrt(365.25 * 24 * 3600)
    else:
        vol = 0.3  # Default
    
    # Handle edge cases quickly
    if T <= 0:
        return 1.0 if (S > strike and option_type == 'call') or (S < strike and option_type == 'put') else 0.0
    
    # Simplified Black-Scholes
    d2 = np.log(S / strike) / (vol * np.sqrt(T))
    
    if option_type.lower() == 'call':
        return norm.cdf(d2) * np.exp(-0.05 * T)
    else:
        return norm.cdf(-d2) * np.exp(-0.05 * T)

def batch_price_options(df, option_chain):
    """
    Price multiple options efficiently
    
    option_chain format:
    [{'strike': 45000, 'expiry_seconds': 1800, 'type': 'call'}, ...]
    """
    results = []
    
    for option in option_chain:
        try:
            result = black_scholes_binary_with_greeks(
                df, 
                option['strike'], 
                option['expiry_seconds'], 
                option['type']
            )
            result.update(option)  # Add original option details
            results.append(result)
        except Exception as e:
            # Fallback pricing
            fallback_price = quick_binary_price(
                df, 
                option['strike'], 
                option['expiry_seconds'], 
                option['type']
            )
            result = option.copy()
            result['price'] = fallback_price
            result['error'] = str(e)
            results.append(result)
    
    return pd.DataFrame(results)

# Usage examples:

df = pd.read_parquet(r"C:\Users\chris\OneDrive\Desktop\Programming\Trading\prediction markets\crypto\brti_data_collection\data\brti_price_data.parquet")
df['log_return'] = np.log(df['brti_price'] / df['brti_price'].shift(1))
df['abs_return'] = np.abs(df['log_return'])
df['abs_return_product'] = df['abs_return'] * df['abs_return'].shift(1)


price = black_scholes_binary(df, 104046, strike=104250, expiry_seconds=time_until_2pm(), option_type='call')
print(f"Binary call price: {price:.4f}")



Binary call price: 0.0878
